In [1]:
from MDN import mdn, Mixture, NLLLoss
from tortreinador.utils.preprocessing import get_dataloader
from tortreinador.train import TorchTrainer, config_generator
from tortreinador.utils.View import init_weights, split_weights
import torch
from tortreinador.utils.metrics import r2_score
from tortreinador.utils.Recorder import MetricManager, MetricDefine
from config import input_parameters, output_parameters, BASE_PATH
import numpy as np
import os

In [ ]:
project_dir = os.path.join(BASE_PATH, 'data')
uncertainty_path = os.path.join(project_dir, '50th')

train_x = np.load(os.path.join(uncertainty_path, "train_x.npy"))
train_y = np.load(os.path.join(uncertainty_path, "train_y.npy"))
val_x = np.load(os.path.join(uncertainty_path, "val_x.npy"))
val_y = np.load(os.path.join(uncertainty_path, "val_y.npy"))

train_dataloader = get_dataloader(torch.as_tensor(train_x).float(), torch.as_tensor(train_y).float(), batch_size=1024, shuffle=False, num_workers=12)
validation_dataloader = get_dataloader(torch.as_tensor(val_x).float(), torch.as_tensor(val_y).float(), batch_size=1024, shuffle=False, num_workers=8)

In [12]:
output_dim = len(output_parameters)
input_dim = len(input_parameters)

model_hps = {
        'input_size': input_dim,
        'output_size': output_dim,
        'num_hidden': 512,
        'num_gaussian': 10,
    }


optim_hps = {
    'lr': 0.0001984,
    'weight_decay': 0.00001
}

dataset_hps = {
    'batch_size': 1024,
    'shuffle': True,
    'num_workers': 0,
    'pin_memory':torch.cuda.is_available()
}

trainer_hps = {
    'epoch': 150,
    'metric_manager': MetricManager([MetricDefine(metric_name='train_loss', metric_mode=1, use_as_criterion=True),
                                     MetricDefine(metric_name='train_r2', metric_mode=1),
                                     MetricDefine(metric_name='val_loss', metric_mode=2, use_as_valloss=True),
                                     MetricDefine(metric_name='val_r2', metric_mode=2, use_as_baseline=True)]),
    'data_save_mode': 'csv'
}
# MetricDefine(metric_name='CD', metric_mode=2)

trainer_configs = {
    'warmup_epochs': 10,
    'best_loss': -1,
    'auto_save': 5,
    'validation_cycle': 1,
    'logger_on': True,
    # 'model_save_path': 'E:\\Resources\\scun',
    # 'logger_dir': 'E:\\Resources\\scun',
    'model_save_path': os.path.join(BASE_PATH, "MDN"),
    'logger_dir': os.path.join(BASE_PATH, "MDN"),
}

In [13]:
# Model
model = mdn(**model_hps)
init_weights(model)
criterion = NLLLoss()
mixture = Mixture()
optimizer = torch.optim.Adam(split_weights(model), **optim_hps)

In [14]:
class Trainer(TorchTrainer):

    def calculate(self, x, y, mode=1):
        pi, mu, sigma = self.model(x)

        loss = self.criterion(pi, mu, sigma, y)

        pdf = mixture(pi, mu, sigma)

        y_pred = pdf.sample()

        r2_result = r2_score(y, y_pred)

        if mode == 1:
            return_dict = {
                        'train_loss': loss,
                        'train_r2': r2_result
                    }

        else:
            return_dict = {
                        'val_loss': loss,
                        'val_r2': r2_result
                    }


        return self._standard_return(mode=mode, update_values=return_dict)

In [15]:
trainer = Trainer(is_gpu=True, model=model, criterion=criterion, optimizer=optimizer, **trainer_hps)
config = config_generator(**trainer_configs)

Epoch:150, Device: cuda


In [ ]:
result = trainer.fit(t_loader, v_loader, **config)